In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
from functools import reduce

from parreg.process_config import load_and_validate_config, get_donors_receivers, process_attr_data
from parreg import utils, utils_algo, funcs_clust, funcs_dist

import time
import os

from parreg.logging_config import setup_logging
setup_logging()

import logging
logger = logging.getLogger(__name__)

In [2]:
# read and validate config
config_file = Path('/home/yuqiong.liu/work/Gitlab/ngen-regionalization/configs/config.yaml')
if not config_file.exists():
    raise FileNotFoundError(config_file)

config = load_and_validate_config(config_file)

2025-05-14 11:58:07,534 - parreg.process_config - INFO - Saving config to /home/yuqiong.liu/work/data/ngen_reg/outputs/run_ngen_hlr/config_final.yaml


In [3]:
# process by VPU
vpu = config.general.vpu_list[0]

In [4]:
# get receivers and qualified donors in the VPU, and compute pairwise distances between them
donors, receivers, df_dist_spatial = get_donors_receivers(config, vpu)

2025-05-14 11:58:13,687 - parreg.config_schema - INFO - Initial donors based on all gages in /home/yuqiong.liu/work/data/ngen_reg/gages_nwm4_calib_all.csv
2025-05-14 11:58:13,722 - parreg.config_schema - INFO - Number of initial donors for vpu 01: 88 gages, 4559 divides
2025-05-14 11:58:13,751 - parreg.config_schema - INFO - Number of donors after filtering: 77 gages, 3649 divides
2025-05-14 11:58:14,016 - parreg.process_config - INFO - Total number of donors in vpu 01: 3649
2025-05-14 11:58:14,016 - parreg.process_config - INFO - Total number of receivers in vpu 01: 500
2025-05-14 11:58:14,017 - parreg.process_config - INFO - Spatial distance file already exists: /home/yuqiong.liu/work/data/ngen_reg/outputs/run_ngen_hlr/spatial_distance/donor_receiver_dist_conus_vpu01.parquet
Skip computing.


In [ ]:
# assemble the attribute data for donors and receivers
df_attrs_all = process_attr_data(config, vpu, donors, receivers, df_dist_spatial)

2025-05-14 11:58:49,409 - parreg.process_config - INFO - Processing attribute data for VPU 01 ... datasets: ['ngen', 'hlr']
2025-05-14 11:58:50,728 - parreg.process_config - WARNING - There are missing data for attributes in vpu 01
2025-05-14 11:58:50,730 - parreg.process_config - INFO - Saving attribute data to /home/yuqiong.liu/work/data/ngen_reg/outputs/run_ngen_hlr/attr_data_final/attr_conus_vpu01.parquet


Number of donors with attribute data: 3649
Number of receivers with attribute data: 500
Missing data percentage for each attribute:
ngen_dksat       0.072307
ngen_psisat      0.072307
hlr_AQPERMNEW    0.433839
hlr_TAVE         0.433839
hlr_PPT          0.433839
hlr_PET          0.433839
hlr_PMPE         0.433839
hlr_SAND         0.433839
dtype: float64


In [ ]:
# detemine whether the catchment is snowy (as snowy and non-snowy catchments are processed separately)
if 'snow_frac' in [col.lower() for col in df_attrs_all.columns]:
    # check if the snow_frac column is present in the attribute data
    n1 = df_attrs_all['snow_frac'].isna().sum()
    if n1 > 0:
        logger.warning(f"There are {n1} missing values in the snow_frac column for VPU {vpu}. Setting them to non-snowy.")
        df_attrs_all['snow_frac'] = df_attrs_all['snow_frac'].fillna(0.0)
    
    # create a new column to indicate whether the catchment is snowy
    df_attrs_all['snowy'] = df_attrs_all['snow_frac'].apply(lambda x: True if x >= config.algorithms.general.min_snow_frac else False)

else:
    # if the snow_frac column is not present, set all catchments to non-snowy
    df_attrs_all['snowy'] = False
    logger.warning(f"The snow_frac column is not present in the attribute data for VPU {vpu}. All catchments are set to non-snowy.")


In [ ]:
# pairing/regionalization algorithms
functions = {
             'proximity': funcs_dist,
             'gower': funcs_dist,
             'urf': funcs_dist,
             'kmeans': funcs_clust,
             'kmedoids': funcs_clust,
             'hdbscan': funcs_clust,
             'birch': funcs_clust,
             }
funcs = functions.keys()

# run only those methods specified to run in the config file
funcs = [x for x in funcs if x in config.general.algorithm_list]

print(f"Algorithms to run: {funcs}")

In [ ]:
# loop through regionalization algorithms and scenarios to generate donor-receiver pairings for each algorithm/scenario combination

for func1 in funcs:
    file_name_str = 'pairs_' + func1 + '_' + config.general.domain + '_vpu' + vpu
    outfile = config.output.pairs.get_file_path(file_name_str)
    
    if outfile.exists():
        logger.info(f'Pair file already exist: {outfile}')
        logger.info(f'Skip the current run: {func1}')
        continue

    logger.info(f'\n======== Identify donors using: {func1}\n') 
    df_donor_all  = pd.DataFrame()   
    start_time = time.time()
    config1 = config.model_dump()['algorithms'][func1]
    config1['njobs'] = config.model_dump()['general']['n_procs']
    config1['non_attr_cols'] = ['divide_id', 'is_donor', 'snowy']
    config1['attrs'] = {'main': [x for x in df_attrs_all.columns if x not in config1['non_attr_cols']],
                        'base': ['ngen_elevation','ngen_slope', 'ngen_aspect']}
    df_donor_all = functions[func1].func(config1, df_attrs_all, df_spatial_dist, func1)  
    end_time = time.time()
    logger.info(f"Execution time: {end_time - start_time:.4f} seconds")
    
    # save donor receiver pairing to csv file    
    config.output.pairs.save_data(df_donor_all, outfile)

In [ ]:
print(df_donor_all)